# Savant 03 fine-tune pilot, second version

This notebook runs the first fine-tuning pilot for the Savant analyst model.

1. It writes verified training conversations from development tables, using seeds far from the held-out bench seed. A scripted analyst writes the code and the answer, and a conversation is kept only when the code ran, every number is a placeholder and no named pair is false.
2. It trains a LoRA adapter on Qwen3-4B-Instruct, learning only the analyst's own turns.
3. It answers 30 bench tables in guided mode with the base model and with the adapter, so both are measured on the same tables in the same session. Blind mode runs on the ten baseline tables with the adapter only.

The second version adds conversations where a first code step fails and the next step fixes it, varies the answer phrasing, lowers the learning rate and enlarges the guided test after the first pilot lost recall.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import subprocess
import sys
import time
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
# The Kaggle image ships an old torchao that peft refuses to load beside, and the pilot
# does not use torchao, so it is removed before peft is imported.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=False)
try:
    import peft  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
import torch  # noqa: E402

TASKS = sorted(glob.glob("/kaggle/input/**/tasks/*.csv", recursive=True))
print(wheel, len(TASKS), "tables")
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Settings for this run.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
TRAIN_TABLES = 400  # development tables tried, each gives one or two conversations
TRACE_SEED = 3_000_000  # far from the bench seed 20261105, which the generator refuses
MAX_LEN = 4096  # longest conversation in tokens, longer ones are skipped
EPOCHS = 1
LR = 5e-5
ACCUM = 8  # conversations per optimizer step
LORA_R, LORA_ALPHA = 16, 32
HELD_OUT = 24  # conversations kept aside to measure loss before and after training
GUIDED_TABLES = {"null": 5, "hard-null": 5, "plant": 20}  # base and adapter on the same tables
BLIND_TABLES = {"null": 3, "hard-null": 3, "plant": 4}  # the ten baseline tables
MAX_NEW_TOKENS = 768
KEEP_MESSAGES = 6
ADAPTER = "/kaggle/working/adapter"

In [ ]:
from savant.traces import PAIR_QUESTION, make_traces, write_traces

t0 = time.time()
records = make_traces(TRAIN_TABLES, TRACE_SEED, workers=os.cpu_count() or 2)
trace_summary = write_traces(records, Path("/kaggle/working/traces.jsonl"))
kept = [r for r in records if r["kept"] and r.get("recall") is not None]
trace_summary["mean_recall_of_kept_planted_traces"] = (
    round(sum(r["recall"] for r in kept) / len(kept), 3) if kept else None
)
trace_summary["seconds"] = round(time.time() - t0)
print(json.dumps(trace_summary, indent=1))

In [ ]:
import random

from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

from savant.finetune import IGNORE, encode_example

tok = AutoTokenizer.from_pretrained(MODEL_ID)


def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(
        messages, add_generation_prompt=add_generation_prompt, tokenize=False, enable_thinking=False
    )


def encode(text):
    return tok(text, add_special_tokens=False)["input_ids"]


examples, skipped = [], {"too_long": 0, "template": 0}
for line in Path("/kaggle/working/traces.jsonl").read_text().splitlines():
    try:
        ex = encode_example(json.loads(line)["messages"], render, encode, MAX_LEN)
    except ValueError:
        skipped["template"] += 1
        continue
    if ex is None:
        skipped["too_long"] += 1
    else:
        examples.append(ex)
assert len(examples) > HELD_OUT + 50, (
    f"too few usable conversations {len(examples)}, skipped {skipped}"
)
random.Random(0).shuffle(examples)
held, train = examples[:HELD_OUT], examples[HELD_OUT:]
lengths = sorted(len(ids) for ids, _ in examples)
print(
    len(train), "train", len(held), "held out", skipped, "median tokens", lengths[len(lengths) // 2]
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()  # adapter weights train in full precision
model.print_trainable_parameters()
base = model.get_base_model()
first = next(base.parameters()).device


def loss_of(ids, labels):
    # Logits are computed only where a label exists, which saves gigabytes on long
    # conversations compared with scoring every position against the full vocabulary.
    x = torch.tensor([ids], device=first)
    y = torch.tensor(labels)
    pos = torch.nonzero(y[1:] != IGNORE).squeeze(1)  # predict token t + 1 from position t
    with torch.autocast("cuda", dtype=torch.float16):
        h = base.model(input_ids=x).last_hidden_state[0]
        logits = base.lm_head(h[pos.to(h.device)]).float()
    return torch.nn.functional.cross_entropy(logits, y[1:][pos].to(logits.device))


@torch.no_grad()
def held_out_loss():
    model.eval()
    vals = [loss_of(i, lab).item() for i, lab in held]
    model.train()
    return round(sum(vals) / len(vals), 4)


before = held_out_loss()
print("held-out loss before", before)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps = max(1, (len(train) * EPOCHS) // ACCUM)
warm = max(1, steps // 20)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, 1 - s / steps)
)
scaler = torch.amp.GradScaler("cuda")
model.train()
t0, step, running, curve = time.time(), 0, [], []
for _epoch in range(EPOCHS):
    for n, (ids, labels) in enumerate(train, 1):
        loss = loss_of(ids, labels) / ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * ACCUM)
        if n % ACCUM == 0 or n == len(train):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            if step % 5 == 0 or step == steps:
                mean = sum(running) / len(running)
                curve.append([step, round(mean, 4)])
                print(f"step {step}/{steps} loss {mean:.4f} {time.time() - t0:.0f}s", flush=True)
                running = []
train_seconds = round(time.time() - t0)
after = held_out_loss()
print("held-out loss after", after)
model.save_pretrained(ADAPTER)
train_summary = {
    "model": MODEL_ID,
    "examples": len(train),
    "held_out": len(held),
    "skipped": skipped,
    "steps": steps,
    "lr": LR,
    "lora_r": LORA_R,
    "held_out_loss_before": before,
    "held_out_loss_after": after,
    "loss_curve": curve,
    "seconds": train_seconds,
}
model = base = opt = None  # free the training copy before evaluation
torch.cuda.empty_cache()

In [ ]:
import gc

import polars as pl

from savant.agent import ask, parse_pairs
from savant.grade import derived_pairs
from savant.llm import Transformers
from savant.planted import Plant, Task


def load_task(csv):
    truth = json.loads(Path(csv.replace(".csv", ".truth.json")).read_text())
    plants = [Plant(p["kind"], tuple(p["columns"]), p["strength"]) for p in truth["plants"]]
    return Task(truth["id"], pl.read_csv(csv, try_parse_dates=True), plants)


tasks = [load_task(c) for c in TASKS]


def pick(counts):
    out = []
    for prefix, count in counts.items():
        out += [t for t in tasks if t.id.startswith(prefix + "-")][:count]
    return out


log = Path("/kaggle/working/runs.jsonl")
log.write_text("")
runs = []


def evaluate(client, model_label, mode, chosen):
    for t in chosen:
        t0 = time.time()
        try:
            r = ask(
                t.table,
                PAIR_QUESTION,
                client,
                keep_messages=KEEP_MESSAGES,
                show_findings=(mode == "guided"),
            )
            error = ""
        except torch.cuda.OutOfMemoryError as e:
            r, error = None, f"out of memory: {e}"[:300]
        torch.cuda.empty_cache()
        real = {frozenset(p.columns) for p in t.plants if p.kind != "leak"}
        leaks = {frozenset(p.columns) for p in t.plants if p.kind == "leak"}
        claimed = parse_pairs(r.rendered if r else "", t.table.columns)
        row = {
            "model": model_label,
            "mode": mode,
            "task": t.id,
            "kind": t.id.rsplit("-", 1)[0],
            "grounded": bool(r and r.grounded),
            "rounds": r.rounds if r else None,
            "steps": len(r.steps) if r else 0,
            "failed_steps": sum(not s.ok for s in r.steps) if r else 0,
            "planted": len(real),
            "found": len(claimed & real),
            "false": len(claimed - real - leaks - derived_pairs(t)),
            "unverified": len(r.unverified_pairs) if r else 0,
            "seconds": round(time.time() - t0, 1),
            "error": error,
            "answer": r.rendered if r else None,
            "transcript": r.transcript if r else None,
        }
        runs.append(row)
        with log.open("a") as f:
            f.write(json.dumps(row, default=str) + "\n")
        print(
            model_label,
            mode,
            t.id,
            "grounded" if row["grounded"] else "NOT grounded",
            row["found"],
            "of",
            row["planted"],
            "found",
            row["false"],
            "false",
            row["seconds"],
            "s",
            error,
            flush=True,
        )


client = Transformers(MODEL_ID, max_new_tokens=MAX_NEW_TOKENS)
evaluate(client, "base", "guided", pick(GUIDED_TABLES))
client = None
gc.collect()
torch.cuda.empty_cache()
client = Transformers(MODEL_ID, max_new_tokens=MAX_NEW_TOKENS, adapter=ADAPTER)
evaluate(client, "adapter", "guided", pick(GUIDED_TABLES))
evaluate(client, "adapter", "blind", pick(BLIND_TABLES))

In [ ]:
def share(xs):
    return round(sum(xs) / len(xs), 3) if xs else None


agent_summary = {"model": MODEL_ID, "tables": {"guided": GUIDED_TABLES, "blind": BLIND_TABLES}}
for label, mode in [("base", "guided"), ("adapter", "guided"), ("adapter", "blind")]:
    rs = [r for r in runs if r["model"] == label and r["mode"] == mode]
    planted = [r for r in rs if r["kind"] == "plant"]
    agent_summary[f"{label} {mode}"] = {
        "tables": len(rs),
        "grounded_answers": share([r["grounded"] for r in rs]),
        "mean_rounds": share([r["rounds"] for r in rs if r["rounds"]]),
        "mean_seconds": share([r["seconds"] for r in rs]),
        "failed_step_share": round(
            sum(r["failed_steps"] for r in rs) / max(1, sum(r["steps"] for r in rs)), 3
        ),
        "planted_pairs": sum(r["planted"] for r in planted),
        "recall": round(
            sum(r["found"] for r in planted) / max(1, sum(r["planted"] for r in planted)), 3
        ),
        "false_per_null_table": share([r["false"] for r in rs if r["kind"] == "null"]),
        "false_per_hard_null_table": share([r["false"] for r in rs if r["kind"] == "hard-null"]),
        "false_per_planted_table": share([r["false"] for r in planted]),
        "unverified_claims": sum(r["unverified"] for r in rs),
        "errors": sum(bool(r["error"]) for r in rs),
    }
Path("/kaggle/working/finetune.json").write_text(
    json.dumps(
        {"traces": trace_summary, "training": train_summary, "agent": agent_summary},
        indent=1,
        default=str,
    )
)
print(json.dumps(agent_summary, indent=1))